In [22]:
from pathlib import Path

import random
import copy

import time
import pandas as pd

!pip install XlsxWriter

#direc=Path().absolute()

#input_path= {
#    "a": Path.joinpath(direc,"data/a_example"),
#    "b": Path.joinpath(direc,"data/b_little_bit_of_everything.in"),
#    "c": Path.joinpath(direc,"data/c_many_ingredients.in"),
#    "d": Path.joinpath(direc,"data/d_many_pizzas.in"),
#    "e": Path.joinpath(direc,"data/e_many_teams.in")
#}

#output_path = {
#    "a": Path.joinpath(direc,"out/a_example.txt"),
#    "b": Path.joinpath(direc,"out/b_little_bit_of_everything.txt"),
#    "c": Path.joinpath(direc,"out/c_many_ingredients.txt"),
#    "d": Path.joinpath(direc,"out/d_many_pizzas.txt"),
#    "e": Path.joinpath(direc,"out/e_many_teams.txt")
#}


data_folder = Path(r'C:\Miguel_Santoalha\Desktop\APE - PSE\2025\Mestrado Engenharia e Ciência de Dados\2025_2026 - 2º Semestre\Inteligência Artificial\Trabalho Grupo\AI_MECD06_Group_G-main\data')

input_path= {
    "a": data_folder / "a_example",
    "b": data_folder / "b_little_bit_of_everything.in",
    "c": data_folder / "c_many_ingredients.in",
    "d": data_folder / "d_many_pizzas.in",
    "e": data_folder / "e_many_teams.in"
}

output_folder = Path(r'C:\Miguel_Santoalha\Desktop\APE - PSE\2025\Mestrado Engenharia e Ciência de Dados\2025_2026 - 2º Semestre\Inteligência Artificial\Trabalho Grupo\AI_MECD06_Group_G-main\out')

input_path = {
    "a": data_folder / "a_example",
    "b": data_folder / "b_little_bit_of_everything.in",
    "c": data_folder / "c_many_ingredients.in",
    "d": data_folder / "d_many_pizzas.in",
    "e": data_folder / "e_many_teams.in"
}

output_path = {
    "a": output_folder / "a_example.txt",
    "b": output_folder / "b_little_bit_of_everything.txt",
    "c": output_folder / "c_many_ingredients.txt",
    "d": output_folder / "d_many_pizzas.txt",
    "e": output_folder / "e_many_teams.txt"
}

print("Welcome to Group G's implementation of Even More Pizzas\n")
print("All datasets (a,b,c,d,e) will be executed automatically.\n")

#print("Welcome to Group G's implementation of ""Even More Pizzas""\n")
#print("The following dataset options are available: a, b, c, d and e \n")

#data=""
#while data not in ["a","b","c","d","e"]:
#    data = input("Please select your data: \n")


parameter_grid = [
    {"iterations": 500, "tabu_size": 25},
    {"iterations": 500, "tabu_size": 50},
    {"iterations": 1000, "tabu_size": 50},
    {"iterations": 1000, "tabu_size": 100}
]
    

for data in ["a","b","c","d","e"]:

    print("\n---------------------------")
    print("Running dataset:",data)
    print("---------------------------")

    pizza_list=[]
    teams={}

    input_loc=input_path[data]
    output_loc=output_path[data]

    
    with open(input_loc) as file:

        for line_index, line in enumerate(file):

            actual_line = line.strip().split()

            if line_index == 0:

                teams = {
                    2: int(actual_line[1]),
                    3: int(actual_line[2]),
                    4: int(actual_line[3])
                }

            else:

                pizza = {
                    "id": line_index - 1,
                    "ingredients": set(actual_line[1:])
                }

                pizza_list.append(pizza)


All datasets (a,b,c,d,e) will be executed automatically.


---------------------------
Running dataset: a
---------------------------


---------------------------
Running dataset: b
---------------------------

---------------------------
Running dataset: c
---------------------------

---------------------------
Running dataset: d
---------------------------

---------------------------
Running dataset: e
---------------------------


In [23]:
def solve_pizza_problem(pizzas, t2, t3, t4):
    # Sort pizzas by number of ingredients (descending) to start with 'richer' options
    pizzas.sort(key=lambda x: len(x['ingredients']), reverse=True)
    
    deliveries = []
    used_pizzas = [False] * len(pizzas)
    
    # Process teams from largest to smallest (4 -> 3 -> 2) 
    # because squares of larger numbers yield higher scores.
    for team_size, team_count in [(4, t4), (3, t3), (2, t2)]:
        for _ in range(team_count):
            current_team_pizzas = []
            current_ingredients = set()
            
            # Fill the team requirements
            for _ in range(team_size):
                best_pizza_idx = -1
                max_score = -1
                
                # Look for the pizza that adds the most value
                for i in range(len(pizzas)):
                    if not used_pizzas[i]:
                        # Calculate how many NEW ingredients this pizza adds
                        new_ingredients = pizzas[i]['ingredients'] - current_ingredients

                        new_count = len(new_ingredients)  

                        diversity_score = new_count * 2 + len(pizzas[i]['ingredients'])


                        if diversity_score > max_score:
                            max_score = diversity_score
                            best_pizza_idx = i
                            
                        # Optimization: if a pizza adds all its ingredients as new, 
                        # it's a strong candidate; we can break early in large datasets.
                
                if best_pizza_idx != -1:
                    used_pizzas[best_pizza_idx] = True
                    current_team_pizzas.append(pizzas[best_pizza_idx]['id'])
                    current_ingredients.update(pizzas[best_pizza_idx]['ingredients'])
                else:
                    # Not enough pizzas left to fill this team
                    break
            
            if len(current_team_pizzas) == team_size:
                deliveries.append((team_size, current_team_pizzas))
            else:
                # Backtrack: if team wasn't filled, mark pizzas as available again
                for p_id in current_team_pizzas:
                    used_pizzas[p_id] = False
                    
    return deliveries

In [24]:
#Score Function

def score(pizzas, deliveries):

    pizza_dict = {p["id"]:p for p in pizzas}

    total_score = 0

    for team_size, pizza_ids in deliveries:

        ingredients=set()

        for pid in pizza_ids:
            ingredients |= pizza_dict[pid]["ingredients"]

        total_score += len(ingredients)**2

    return total_score

In [25]:
#def score(pizzas,deliveries):
#    score_list=[]
#    for i in range(len(deliveries)):
#        delivered_pizzas=deliveries[i][1]
#        ingredients=[]
#        for id in delivered_pizzas:
#            ingredient_list=pizzas[id]['ingredients']
#            for ingredient in ingredient_list:
#                ingredients.append(ingredient)
#        unique_ingredients=set(ingredients)
#        score_list.append(len(unique_ingredients)**2)
#        #print(score_list)
#    if len(deliveries)==1:
#        total_score=score_list[0]
#    else:
#        total_score=sum(score_list)
#    return total_score

In [26]:
# Tabu Search

def tabu_search(pizzas, deliveries, iterations, tabu_size):

#def tabu_search(pizzas, deliveries, iterations=3000, tabu_size=100):

    pizza_dict = {p["id"]:p for p in pizzas}

    best_solution = copy.deepcopy(deliveries)
    current_solution = copy.deepcopy(deliveries)

    best_score = score(pizzas,best_solution)

    tabu_list=[]

    all_pizzas=set(pizza_dict.keys())

    for it in range(iterations):

        candidate = copy.deepcopy(current_solution)

        move_type=random.choice(["swap","replace"])

        # ---------------------------------
        # Assuming SWAP pizzas between teams
        # ---------------------------------

        if move_type=="swap" and len(candidate)>=2:

            d1,d2=random.sample(range(len(candidate)),2)

            team1,pizzas1=candidate[d1]
            team2,pizzas2=candidate[d2]

            i=random.randrange(len(pizzas1))
            j=random.randrange(len(pizzas2))

            move=(pizzas1[i],pizzas2[j])

            pizzas1[i],pizzas2[j]=pizzas2[j],pizzas1[i]

            candidate[d1]=(team1,pizzas1)
            candidate[d2]=(team2,pizzas2)

        # --------------------------------------------------------
        # Assuming the possibility if replacing with unused pizza
        # --------------------------------------------------------

        else:

            used=set(p for _,plist in candidate for p in plist)
            unused=list(all_pizzas-used)

            if not unused:
                continue

            d=random.randrange(len(candidate))
            team,pizza_ids=candidate[d]

            idx=random.randrange(len(pizza_ids))

            new_pizza=random.choice(unused)

            move=(pizza_ids[idx],new_pizza)

            pizza_ids[idx]=new_pizza

            candidate[d]=(team,pizza_ids)

        candidate_score=score(pizzas,candidate)

        # Aspiration criterion
        if move in tabu_list and candidate_score<=best_score:
            continue

        current_solution=candidate

        if candidate_score>best_score:

            best_solution=copy.deepcopy(candidate)
            best_score=candidate_score

        tabu_list.append(move)

        if len(tabu_list)>tabu_size:
            tabu_list.pop(0)

        if it % max(1, iterations//10) == 0:
            print("Iteration",it,"Best score:",best_score)

    return best_solution

In [27]:
# Output

def output_write(output_path,deliveries):
    with open(output_path, "w") as file:
        file.write(str(len(deliveries)))
        file.write("\n")
        for team_size, pizza_id in deliveries:
            line = " ".join(map(str, [team_size] + pizza_id))
            file.write(line + "\n")

In [28]:
# Pipeline for execution of the code

results = []

for data in ["a", "b", "c", "d", "e"]:

    print("\n==============================")
    print("Running dataset:", data)
    print("==============================")

    pizza_list = []
    teams = {}

    input_loc = input_path[data]
    output_loc = output_path[data]

    with open(input_loc) as file:

        for line_index, line in enumerate(file):

            actual_line = line.strip().split()

            if line_index == 0:

                teams = {
                    2: int(actual_line[1]),
                    3: int(actual_line[2]),
                    4: int(actual_line[3])
                }

            else:

                pizza = {
                    "id": line_index - 1,
                    "ingredients": set(actual_line[1:])
                }

                pizza_list.append(pizza)

    print("Loaded pizzas:", len(pizza_list))

    start_time = time.time()

    deliveries = solve_pizza_problem(pizza_list, teams[2], teams[3], teams[4])

    initial_score = score(pizza_list, deliveries)

    greedy_time = time.time() - start_time

    print("Initial greedy score:", initial_score)

    results.append({
        "dataset": data,
        "algorithm": "Greedy",
        "iterations": 0,
        "tabu_size": 0,
        "score": initial_score,
        "runtime_seconds": greedy_time
    })

    for params in parameter_grid:

        print("\nTesting parameters:", params)

        start_time = time.time()

        best_solution = tabu_search(
            pizza_list,
            deliveries,
            iterations=params["iterations"],
            tabu_size=params["tabu_size"]
        )

        final_score = score(pizza_list, best_solution)

        tabu_time = time.time() - start_time

        print("Final score:", final_score)

        results.append({
            "dataset": data,
            "algorithm": "Tabu Search",
            "iterations": params["iterations"],
            "tabu_size": params["tabu_size"],
            "score": final_score,
            "runtime_seconds": tabu_time,
            "improvement_vs_greedy": final_score - initial_score
        })

    output_write(output_loc, best_solution)

# -----------------------------
# Export results to Excel
# -----------------------------

results_df = pd.DataFrame(results)

excel_output = output_folder / "experiment_results.xlsx"

score_table = results_df.pivot_table(
    index="dataset",
    columns="algorithm",
    values="score",
    aggfunc="max"
)

time_table = results_df.pivot_table(
    index="dataset",
    columns="algorithm",
    values="runtime_seconds",
    aggfunc="mean"
)

parameter_table = results_df.pivot_table(
    index=["iterations", "tabu_size"],
    values="score",
    aggfunc="mean"
)

with pd.ExcelWriter(excel_output, engine="xlsxwriter") as writer:

    results_df.to_excel(writer, sheet_name="Results", index=False)

    workbook = writer.book

    parameter_table.to_excel(writer, sheet_name="Parameter Analysis")
    score_table.to_excel(writer, sheet_name="Score Table")
    time_table.to_excel(writer, sheet_name="Runtime Table")

    score_ws = writer.sheets["Score Table"]
    time_ws = writer.sheets["Runtime Table"]

    score_chart = workbook.add_chart({"type": "column"})

    score_chart.add_series({
        "name": "Greedy",
        "categories": ["Score Table", 1, 0, len(score_table), 0],
        "values": ["Score Table", 1, 1, len(score_table), 1],
    })

    score_chart.add_series({
        "name": "Tabu Search",
        "categories": ["Score Table", 1, 0, len(score_table), 0],
        "values": ["Score Table", 1, 2, len(score_table), 2],
    })

    score_chart.set_title({"name": "Score Comparison"})
    score_chart.set_x_axis({"name": "Dataset"})
    score_chart.set_y_axis({"name": "Score"})

    score_ws.insert_chart("E2", score_chart)

    runtime_chart = workbook.add_chart({"type": "column"})

    runtime_chart.add_series({
        "name": "Greedy",
        "categories": ["Runtime Table", 1, 0, len(time_table), 0],
        "values": ["Runtime Table", 1, 1, len(time_table), 1],
    })

    runtime_chart.add_series({
        "name": "Tabu Search",
        "categories": ["Runtime Table", 1, 0, len(time_table), 0],
        "values": ["Runtime Table", 1, 2, len(time_table), 2],
    })

    runtime_chart.set_title({"name": "Runtime Comparison"})
    runtime_chart.set_x_axis({"name": "Dataset"})
    runtime_chart.set_y_axis({"name": "Seconds"})

    time_ws.insert_chart("E2", runtime_chart)

print("\nResults exported to:", excel_output)




Running dataset: a
Loaded pizzas: 5
Initial greedy score: 49

Testing parameters: {'iterations': 500, 'tabu_size': 25}
Iteration 0 Best score: 49
Final score: 49

Testing parameters: {'iterations': 500, 'tabu_size': 50}
Iteration 0 Best score: 49
Final score: 49

Testing parameters: {'iterations': 1000, 'tabu_size': 50}
Iteration 0 Best score: 49
Final score: 49

Testing parameters: {'iterations': 1000, 'tabu_size': 100}
Iteration 0 Best score: 49
Final score: 49

Running dataset: b
Loaded pizzas: 500
Initial greedy score: 7280

Testing parameters: {'iterations': 500, 'tabu_size': 25}
Iteration 0 Best score: 7325
Iteration 150 Best score: 7367
Iteration 200 Best score: 7367
Iteration 300 Best score: 7367
Iteration 450 Best score: 7367
Final score: 7367

Testing parameters: {'iterations': 500, 'tabu_size': 50}
Iteration 50 Best score: 7280
Iteration 100 Best score: 7280
Iteration 200 Best score: 7280
Iteration 350 Best score: 7280
Iteration 450 Best score: 7320
Final score: 7320

Testi

In [29]:
#deliveries=solve_pizza_problem(pizza_list, teams[2], teams[3], teams[4])
#sco=score(pizza_list,deliveries)
#print(deliveries)
#print(sco)
#output_write(output_loc,deliveries)

In [30]:
''' Local
a: 49 (< 1 segundo)
b: 6502 (< 1 segundo)
c: 229352265 (3-4 minutos)
d: 2046959 (30-40 minutos)
e: 8257534 (1 hora)
'''

' Local\na: 49 (< 1 segundo)\nb: 6502 (< 1 segundo)\nc: 229352265 (3-4 minutos)\nd: 2046959 (30-40 minutos)\ne: 8257534 (1 hora)\n'